# 📈 Stock Revenue Dashboard — Tesla & GameStop

Dự án này mình thực hiện để thực hành **thu thập dữ liệu chứng khoán** kết hợp **web scraping** rồi vẽ dashboard so sánh giá cổ phiếu và doanh thu theo quý.

Hai công ty mình chọn phân tích:
- **Tesla (TSLA)** — ông lớn xe điện
- **GameStop (GME)** — cổ phiếu "meme" nổi tiếng năm 2021

---

## 🗂️ Nội dung notebook

1. [Cài đặt thư viện](#setup)
2. [Hàm vẽ đồ thị](#graph)
3. [Lấy dữ liệu giá cổ phiếu Tesla (yfinance)](#q1)
4. [Scraping doanh thu Tesla](#q2)
5. [Lấy dữ liệu giá cổ phiếu GameStop (yfinance)](#q3)
6. [Scraping doanh thu GameStop](#q4)
7. [Dashboard Tesla](#q5)
8. [Dashboard GameStop](#q6)

<a id='setup'></a>
## ⚙️ 1. Cài đặt & Import thư viện

In [ ]:
# Cài các thư viện cần thiết (chỉ cần chạy 1 lần)
# Bỏ comment dòng nào chưa có trong môi trường của bạn
# !pip install yfinance==0.2.38 --quiet
# !pip install beautifulsoup4 --quiet
# !pip install lxml --quiet
# !pip install matplotlib --quiet

In [ ]:
import yfinance as yf
import pandas as pd
import requests
from bs4 import BeautifulSoup
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

print("✅ Import xong hết rồi, bắt đầu thôi!")

<a id='graph'></a>
## 📊 2. Định nghĩa hàm vẽ đồ thị

Hàm `make_graph` nhận vào:
- `stock_data`: DataFrame chứa cột `Date` và `Close` (giá đóng cửa)
- `revenue_data`: DataFrame chứa cột `Date` và `Revenue` (doanh thu theo quý, đơn vị triệu USD)
- `stock`: tên công ty để làm tiêu đề đồ thị

In [ ]:
def make_graph(stock_data, revenue_data, stock):
    """Vẽ dashboard 2 panel: giá cổ phiếu (trên) và doanh thu theo quý (dưới).
    
    Dữ liệu chỉ hiển thị tới tháng 6/2021 để khớp với dữ liệu scraping.
    """
    # Lọc dữ liệu tới mốc thời gian có dữ liệu đủ cả 2 nguồn
    stock_data_specific = stock_data[stock_data.Date <= '2021-06-14'].copy()
    revenue_data_specific = revenue_data[revenue_data.Date <= '2021-04-30'].copy()

    fig, axes = plt.subplots(2, 1, figsize=(14, 9))
    fig.suptitle(f"{stock} — Stock Price & Revenue Dashboard", fontsize=16, fontweight='bold', y=1.01)

    # Panel trên: giá cổ phiếu
    axes[0].plot(
        pd.to_datetime(stock_data_specific.Date),
        stock_data_specific.Close.astype(float),
        color='#1f77b4', linewidth=1.5, label='Giá đóng cửa'
    )
    axes[0].fill_between(
        pd.to_datetime(stock_data_specific.Date),
        stock_data_specific.Close.astype(float),
        alpha=0.15, color='#1f77b4'
    )
    axes[0].set_ylabel("Giá ($USD)", fontsize=12)
    axes[0].set_title(f"{stock} — Lịch sử giá cổ phiếu", fontsize=13)
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    axes[0].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

    # Panel dưới: doanh thu theo quý
    axes[1].bar(
        pd.to_datetime(revenue_data_specific.Date),
        revenue_data_specific.Revenue.astype(float),
        width=60, color='#2ca02c', alpha=0.8, label='Doanh thu'
    )
    axes[1].set_ylabel("Doanh thu (triệu $USD)", fontsize=12)
    axes[1].set_xlabel("Năm", fontsize=12)
    axes[1].set_title(f"{stock} — Doanh thu theo quý", fontsize=13)
    axes[1].legend()
    axes[1].grid(True, alpha=0.3, axis='y')
    axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

    plt.tight_layout()
    plt.savefig(f"{stock.lower().replace(' ', '_')}_dashboard.png", dpi=150, bbox_inches='tight')
    plt.show()
    print(f"💾 Đã lưu ảnh: {stock.lower().replace(' ', '_')}_dashboard.png")

<a id='q1'></a>
## 🚗 3. Lấy dữ liệu giá cổ phiếu Tesla (yfinance)

Dùng thư viện `yfinance` để lấy toàn bộ lịch sử giá cổ phiếu của Tesla (ticker: `TSLA`).

In [ ]:
# Tạo ticker object cho Tesla
tesla_ticker = yf.Ticker("TSLA")

# Lấy toàn bộ lịch sử giá (period='max' = từ ngày đầu tiên đến nay)
tesla_data = tesla_ticker.history(period="max")

# Reset index để cột Date trở thành cột bình thường (không phải index)
tesla_data.reset_index(inplace=True)

# Chuyển Date sang dạng string cho nhất quán khi filter
tesla_data['Date'] = tesla_data['Date'].astype(str).str[:10]

print(f"📦 Tesla data: {tesla_data.shape[0]} rows, {tesla_data.shape[1]} columns")
print(f"📅 Từ {tesla_data['Date'].min()} đến {tesla_data['Date'].max()}")
tesla_data.head()

<a id='q2'></a>
## 💰 4. Scraping doanh thu Tesla

Lấy dữ liệu doanh thu theo quý của Tesla từ trang web bằng `requests` + `BeautifulSoup`.

In [ ]:
# URL chứa bảng doanh thu Tesla theo quý
url_tesla = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-PY0220EN-SkillsNetwork/labs/project/revenue.htm"

# Gửi request lấy HTML về
html_data = requests.get(url_tesla).text
print(f"✅ Lấy HTML thành công, độ dài: {len(html_data)} ký tự")

In [ ]:
# Parse HTML bằng BeautifulSoup
soup_tesla = BeautifulSoup(html_data, "html.parser")

# Tìm bảng doanh thu — bảng thứ 2 trong trang (index 1)
tesla_revenue = pd.DataFrame(columns=["Date", "Revenue"])

table = soup_tesla.find_all("tbody")[1]   # bảng Tesla Quarterly Revenue
rows = table.find_all("tr")

for row in rows:
    cols = row.find_all("td")
    if len(cols) >= 2:
        date = cols[0].text.strip()
        revenue = cols[1].text.strip()
        tesla_revenue = pd.concat(
            [tesla_revenue, pd.DataFrame({"Date": [date], "Revenue": [revenue]})],
            ignore_index=True
        )

print(f"📦 Scraped xong: {len(tesla_revenue)} dòng")

In [ ]:
# Làm sạch cột Revenue: bỏ dấu $, dấu phẩy
tesla_revenue["Revenue"] = tesla_revenue['Revenue'].str.replace(',|\$', "", regex=True)

# Bỏ các dòng null hoặc chuỗi rỗng
tesla_revenue.dropna(inplace=True)
tesla_revenue = tesla_revenue[tesla_revenue['Revenue'] != ""]

print("✅ Làm sạch xong!")
print(f"📦 Còn lại: {len(tesla_revenue)} dòng")
tesla_revenue.tail()

<a id='q3'></a>
## 🎮 5. Lấy dữ liệu giá cổ phiếu GameStop (yfinance)

Tương tự Tesla nhưng với GameStop (ticker: `GME`) — cổ phiếu nổi tiếng vì cộng đồng Reddit đẩy giá năm 2021.

In [ ]:
# Tạo ticker object cho GameStop
gme_ticker = yf.Ticker("GME")

# Lấy toàn bộ lịch sử giá
gme_data = gme_ticker.history(period="max")

# Reset index
gme_data.reset_index(inplace=True)

# Chuyển Date sang string
gme_data['Date'] = gme_data['Date'].astype(str).str[:10]

print(f"📦 GameStop data: {gme_data.shape[0]} rows, {gme_data.shape[1]} columns")
print(f"📅 Từ {gme_data['Date'].min()} đến {gme_data['Date'].max()}")
gme_data.head()

<a id='q4'></a>
## 💸 6. Scraping doanh thu GameStop

Lấy doanh thu theo quý của GameStop từ trang web.

In [ ]:
# URL chứa bảng doanh thu GameStop
url_gme = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-PY0220EN-SkillsNetwork/labs/project/stock.html"

# Gửi request
html_data_2 = requests.get(url_gme).text
print(f"✅ Lấy HTML thành công, độ dài: {len(html_data_2)} ký tự")

In [ ]:
# Parse HTML
soup_gme = BeautifulSoup(html_data_2, "html.parser")

# Tìm bảng doanh thu GameStop
gme_revenue = pd.DataFrame(columns=["Date", "Revenue"])

table_gme = soup_gme.find_all("tbody")[1]
rows_gme = table_gme.find_all("tr")

for row in rows_gme:
    cols = row.find_all("td")
    if len(cols) >= 2:
        date = cols[0].text.strip()
        revenue = cols[1].text.strip()
        gme_revenue = pd.concat(
            [gme_revenue, pd.DataFrame({"Date": [date], "Revenue": [revenue]})],
            ignore_index=True
        )

print(f"📦 Scraped xong: {len(gme_revenue)} dòng")

In [ ]:
# Làm sạch dữ liệu doanh thu GameStop
gme_revenue["Revenue"] = gme_revenue['Revenue'].str.replace(',|\$', "", regex=True)

gme_revenue.dropna(inplace=True)
gme_revenue = gme_revenue[gme_revenue['Revenue'] != ""]

print("✅ Làm sạch xong!")
print(f"📦 Còn lại: {len(gme_revenue)} dòng")
gme_revenue.tail()

<a id='q5'></a>
## 📉 7. Dashboard Tesla

Vẽ dashboard tổng hợp cho Tesla — giá cổ phiếu và doanh thu theo quý.

In [ ]:
make_graph(tesla_data, tesla_revenue, 'Tesla')

<a id='q6'></a>
## 🕹️ 8. Dashboard GameStop

Vẽ dashboard tổng hợp cho GameStop — có thể thấy rõ cú bơm giá kinh điển đầu năm 2021.

In [ ]:
make_graph(gme_data, gme_revenue, 'GameStop')

---

## 🔍 Nhận xét nhanh

**Tesla:**
- Giá cổ phiếu tăng mạnh từ cuối 2019 đến đầu 2021, phản ánh kỳ vọng cao vào xe điện
- Doanh thu tăng trưởng đều đặn và ổn định theo từng quý — nền tảng kinh doanh thực chất

**GameStop:**
- Doanh thu thực tế đang đi xuống (mô hình kinh doanh game vật lý bị lỗi thời)
- Giá cổ phiếu bơm lên cực mạnh đầu 2021 do hiện tượng short squeeze từ cộng đồng r/WallStreetBets
- Đây là ví dụ điển hình cho thấy **giá cổ phiếu không phải lúc nào cũng phản ánh tình hình kinh doanh thực**

---

## 📚 Công nghệ sử dụng

| Thư viện | Mục đích |
|---|---|
| `yfinance` | Lấy dữ liệu giá cổ phiếu từ Yahoo Finance |
| `requests` | Gửi HTTP request để lấy trang web |
| `BeautifulSoup` | Parse HTML và trích xuất dữ liệu bảng |
| `pandas` | Xử lý và làm sạch dữ liệu |
| `matplotlib` | Vẽ đồ thị |
